<a href="https://colab.research.google.com/github/iris-hep/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_7_Asimov_Misspecification_Coverage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 7 — Asimov closure and coverage under model misspecification

Exercise 5 constructed a tractable hybrid neural density estimation (hNDE) model. This exercise asks what remains true if its process-to-reference ratios are deliberately made very wrong.

Two statements must be separated:

1. **Model-relative Asimov closure.** If positive process ratios are normalized on the same weighted reference sample used to construct the Asimov dataset, the finite-sample score for yield-linear parameters vanishes exactly at the generating point. This algebraic property does not require the ratios to resemble the physical truth.
2. **Frequentist coverage.** Confidence intervals have their advertised coverage only for repeated data generated by the statistical model used to calibrate them. Asimov closure alone cannot guarantee coverage when real data are generated by a different distribution.

Both statements are demonstrated. A deliberately misspecified model is first used consistently for its Asimov sample, likelihood, and hNDE-based toys. It closes and is described by its own asymptotic approximation, although its expected likelihood scan is less powerful. The same bad likelihood is then applied to toys drawn from a fresh, independent simulator bank. This external comparison exposes the inference failure that can remain under genuine misspecification.

The expensive Exercise 5 PRESEL classifier, reference flow, and ratio ensembles are loaded rather than retrained. The independent simulator bank is cached after the frozen networks are evaluated once. Every figure is exported as a self-contained script in exercise7_figures_scripts/.


In [ ]:
# Google Colab setup — safe to re-run and a no-op off Colab.
import os, sys

REPO_URL = "https://github.com/iris-hep/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
N_BKG, N_SIG = 100_000_000, 20_000_000
USE_DRIVE = True
REMAKE_EVENTS = False
# ----------------------------------------------------------------------------

import subprocess
from pathlib import Path

DEPENDENCIES = [
    "pytorch-lightning",
    "onnx",
    "onnxruntime",
    "onnxscript",
    "iminuit",
    "mplhep",
    "nflows",
    "pyarrow",
]


def run(*args, env=None):
    """Run one setup command and stop immediately if it fails."""
    subprocess.run([str(arg) for arg in args], check=True, env=env)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab")
    else:
        ROOT = Path("/content")
    ROOT.mkdir(parents=True, exist_ok=True)

    # Keep the checkout where earlier notebook versions placed it. The source
    # helpers now live in ml4hep_tifr_colab, while the untracked legacy
    # ml4hep_tifr directory remains the persistent workspace for data/models.
    REPO_DIR = ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    WORK_DIR = REPO_DIR / "workshops" / "ml4hep_tifr"

    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)

    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )

    # Import package code and tutorial-local helpers from their new locations.
    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        import_path = str(import_dir.resolve())
        if import_path not in sys.path:
            sys.path.insert(0, import_path)

    run(sys.executable, "-m", "pip", "install", "-q", *DEPENDENCIES)

    # Preserve the exact old working location. Existing dataframes, models,
    # densities, and plots are reused; only a missing directory is created.
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)

    remake_events = globals().get("REMAKE_EVENTS", False)
    if remake_events or not Path("dataframes/signal.parquet").exists():
        run(
            sys.executable,
            TUTORIAL_DIR / "generate_distributions.py",
            "--n_bkg", N_BKG,
            "--n_sig", N_SIG,
        )

print("Working dir:", os.getcwd())


In [ ]:
import gc
from pathlib import Path

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
from scipy.stats import chi2, ncx2, norm
import torch

from nsbi_common_utils.training.utils import load_trained_model
from utils import FEATURES, predict_with_model
from utils_distributions import (
    background_components,
    signal_components,
    smearing_parameters,
)
from utils_nf import (
    accumulate_preselection_histogram,
    checkpoint_path,
    choose_preselection_ratio_cut,
    flow_sample_x,
    load_flow,
)
from utils_plotting import export_standalone_figure_script

FEATURES = list(FEATURES)
N_DIM = len(FEATURES)
SEED = 12345
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Load the frozen Exercise 5 model

The checkpoint paths and PRESEL definition are identical to Exercises 5 and 6. A new one-million-event reference sample is drawn from the saved flow. This is sufficient for the misspecification demonstration because no network is trained here.


In [ ]:
BASE_PATH = Path("./dataframes")
PRESEL_MODEL_DIR = Path("models_PRESEL")
REFERENCE_FLOW_MODEL_DIR = Path("models_flows_hybrid_reference_spline16_tail5")
RATIO_MODEL_DIR = {
    "signal": Path("models_Hybrid_SigvsRef_5M_ensemble4"),
    "background": Path("models_Hybrid_BkgvsRef_5M_ensemble4"),
}
OUTPUT_DIR = Path("saved_exercise7_misspecification")
PLOT_DIR = Path("plots_exercise7_misspecification")
FIGURE_SCRIPT_DIR = Path("exercise7_figures_scripts")
SIMULATOR_BANK_DIR = OUTPUT_DIR / "simulator_q_banks"
for directory in [
    OUTPUT_DIR,
    PLOT_DIR,
    FIGURE_SCRIPT_DIR,
    SIMULATOR_BANK_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

SAMPLE_PATHS = {
    "signal": BASE_PATH / "signal.parquet",
    "background": BASE_PATH / "background.parquet",
}
SPLIT_SEED = 0
PRESEL_TRAIN_FRACTION = 0.5
FLOW_TRAIN_FRACTION = 0.92
STREAM_BATCH_SIZE = 100_000
PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO = 250.0
PRESEL_CUT_HISTOGRAM_BINS = 4_000
PRESEL_LOG_RATIO_RANGE = (-20.0, 20.0)
REFERENCE_FLOW_TYPE = "quadratic_spline"
REFERENCE_SAMPLING_BATCH_SIZE = 65_536
RATIO_ENSEMBLE_SIZE = 4
RATIO_EVALUATION_BATCH_SIZE = 100_000
RATIO_FLOOR = 1.0e-12
N_REFERENCE_EVENTS = 1_000_000

ASIMOV_MU_TRUE = 1.0
MU_SCAN = np.linspace(0.0, 3.0, 121)
# The bad signal is modeled mostly as the reference, with a small admixture
# of the original background; the bad background is defined conversely.
DEFORMATION_DILUTION = 0.25

SIMULATOR_BANK_SELECTED_EVENTS = 5_000_000
SIMULATOR_BANK_GENERATION_BATCH_SIZE = 250_000
SIMULATOR_BANK_SEED = 271828
SIMULATOR_BANK_FORCE_REBUILD = False

N_TOYS = 100_000
TOY_Q_BINS = 512
TOY_BATCH_SIZE = 2_000
TOY_NEWTON_STEPS = 16
TOY_MU_MAX = 12.0
COVERAGE_LEVELS = np.asarray([0.68, 0.90, 0.95])
MODEL_GENERATOR_LABEL = "Misspecified hNDE model"
SIMULATOR_GENERATOR_LABEL = "Independent simulator"


def export_exercise7_figure(fig, script_name):
    return export_standalone_figure_script(
        fig, script_name=script_name, output_dir=FIGURE_SCRIPT_DIR
    )


required_paths = [
    PRESEL_MODEL_DIR / "model0.onnx",
    PRESEL_MODEL_DIR / "model_scaler0.bin",
    checkpoint_path("reference", REFERENCE_FLOW_MODEL_DIR, REFERENCE_FLOW_TYPE),
]
for sample_name, model_dir in RATIO_MODEL_DIR.items():
    for member in range(RATIO_ENSEMBLE_SIZE):
        required_paths.extend(
            [model_dir / f"model{member}.onnx", model_dir / f"model_scaler{member}.bin"]
        )
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Exercise 7 loads Exercise 5 checkpoints. Missing:\n"
        + "\n".join(f"  - {path}" for path in missing_paths)
    )
print(f"Standalone figure scripts will be written to {FIGURE_SCRIPT_DIR}/")


In [ ]:
def as_inference_session(model_candidate):
    if isinstance(model_candidate, ort.InferenceSession):
        return model_candidate
    available = ort.get_available_providers()
    providers = [
        provider
        for provider in ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if provider in available
    ] or available
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    options.inter_op_num_threads = 1
    return ort.InferenceSession(
        model_candidate.SerializeToString(),
        sess_options=options,
        providers=providers,
    )


PRESEL_scaler, PRESEL_model_proto = load_trained_model(
    PRESEL_MODEL_DIR / "model0.onnx", PRESEL_MODEL_DIR / "model_scaler0.bin"
)
PRESEL_model = as_inference_session(PRESEL_model_proto)
del PRESEL_model_proto


def evaluate_PRESEL_ratio(feature_dataframe):
    ratio = predict_with_model(
        feature_dataframe.astype("float32", copy=False),
        scaler=PRESEL_scaler,
        model=PRESEL_model,
    )
    return np.asarray(ratio, dtype=np.float64).reshape(-1)


PRESEL_STATE_CANDIDATES = [
    OUTPUT_DIR / "exercise5_preselection_state.npz",
    Path("saved_asimov_nis_influence_v2/exercise5_preselection_state.npz"),
    Path("saved_asimov_nis_scan_v1/exercise5_preselection_state.npz"),
]
existing_state = next((path for path in PRESEL_STATE_CANDIDATES if path.exists()), None)
if existing_state is not None:
    state = np.load(existing_state)
    PRESEL_RATIO_CUT = float(state["ratio_cut"])
    LAM_SIG = float(state["lambda_signal"])
    LAM_BKG = float(state["lambda_background"])
    print(f"Loaded PRESEL state from {existing_state}")
else:
    edges = np.linspace(
        PRESEL_LOG_RATIO_RANGE[0], PRESEL_LOG_RATIO_RANGE[1],
        PRESEL_CUT_HISTOGRAM_BINS + 1,
    )
    histograms, statistics = {}, {}
    for sample_name in ["signal", "background"]:
        histograms[sample_name], statistics[sample_name] = (
            accumulate_preselection_histogram(
                SAMPLE_PATHS[sample_name],
                features=FEATURES, ratio_predictor=evaluate_PRESEL_ratio,
                log_ratio_edges=edges, batch_size=STREAM_BATCH_SIZE,
                presel_fraction=PRESEL_TRAIN_FRACTION,
                flow_train_fraction=FLOW_TRAIN_FRACTION, split_seed=SPLIT_SEED,
            )
        )
    PRESEL_RATIO_CUT, diagnostics = choose_preselection_ratio_cut(
        histograms["signal"], histograms["background"], edges,
        signal_inclusive_yield=statistics["signal"]["inclusive_weight"],
        background_inclusive_yield=statistics["background"]["inclusive_weight"],
        signal_partition_weight=statistics["signal"]["partition_weight"],
        background_partition_weight=statistics["background"]["partition_weight"],
        target_background_to_signal=PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO,
    )
    LAM_SIG = diagnostics["histogram_signal_yield"]
    LAM_BKG = diagnostics["histogram_background_yield"]
    np.savez(
        PRESEL_STATE_CANDIDATES[0], ratio_cut=PRESEL_RATIO_CUT,
        lambda_signal=LAM_SIG, lambda_background=LAM_BKG,
    )

reference_flow = load_flow(
    "reference", model_dir=REFERENCE_FLOW_MODEL_DIR,
    flow_type=REFERENCE_FLOW_TYPE, device=device, expected_features=FEATURES,
)
ratio_models = {}
for sample_name, model_dir in RATIO_MODEL_DIR.items():
    ratio_models[sample_name] = []
    for member in range(RATIO_ENSEMBLE_SIZE):
        scaler, model_proto = load_trained_model(
            model_dir / f"model{member}.onnx",
            model_dir / f"model_scaler{member}.bin",
        )
        ratio_models[sample_name].append(
            {"scaler": scaler, "model": as_inference_session(model_proto)}
        )
print(f"Post-selection yields: signal={LAM_SIG:.6g}, background={LAM_BKG:.6g}")


## Draw the reference sample and evaluate the original ratios

The same PRESEL boundary is imposed by rejection sampling. The four ensemble predictions are averaged as ratios and normalized on this new reference sample. These normalized ratios define the original Exercise 5 model and provide the ingredients for the controlled deformation below. They are not used as a substitute for simulator-based toys in the final closure test.


In [ ]:
def evaluate_ratio(sample_name, values, batch_size=RATIO_EVALUATION_BATCH_SIZE):
    values = np.asarray(values, dtype=np.float32)
    chunks = []
    for start in range(0, len(values), int(batch_size)):
        batch = pd.DataFrame(values[start:start + int(batch_size)], columns=FEATURES)
        member_predictions = []
        for pack in ratio_models[sample_name]:
            prediction = predict_with_model(
                batch, scaler=pack["scaler"], model=pack["model"]
            )
            member_predictions.append(
                np.asarray(prediction, dtype=np.float64).reshape(-1)
            )
        chunks.append(np.mean(np.stack(member_predictions, axis=0), axis=0))
    ratio = np.concatenate(chunks)
    if not np.isfinite(ratio).all():
        raise FloatingPointError(f"Non-finite {sample_name} ratio.")
    return np.maximum(ratio, RATIO_FLOOR)


def sample_preselected_flow(flow_pack, n_events, batch_size=65_536):
    accepted_chunks = []
    n_kept = 0
    while n_kept < int(n_events):
        needed = int(n_events) - n_kept
        current_batch = max(int(batch_size), min(4 * int(batch_size), 2 * needed))
        generated = flow_sample_x(flow_pack, current_batch, batch_size=batch_size)
        generated_df = pd.DataFrame(generated, columns=FEATURES)
        passes = evaluate_PRESEL_ratio(generated_df) >= PRESEL_RATIO_CUT
        if np.any(passes):
            accepted_chunks.append(generated[passes])
            n_kept += int(passes.sum())
    return np.concatenate(accepted_chunks, axis=0)[:int(n_events)].astype(np.float32)


torch.manual_seed(SEED + 700)
reference_values = sample_preselected_flow(
    reference_flow, N_REFERENCE_EVENTS, REFERENCE_SAMPLING_BATCH_SIZE
)
raw_signal = evaluate_ratio("signal", reference_values)
raw_background = evaluate_ratio("background", reference_values)
GOOD_RATIO_NORMALIZATION = {
    "signal": float(raw_signal.mean()),
    "background": float(raw_background.mean()),
}
ratio_signal_good = raw_signal / GOOD_RATIO_NORMALIZATION["signal"]
ratio_background_good = raw_background / GOOD_RATIO_NORMALIZATION["background"]
weight_signal_good = ratio_signal_good / N_REFERENCE_EVENTS
weight_background_good = ratio_background_good / N_REFERENCE_EVENTS
print("Original ratio means:", ratio_signal_good.mean(), ratio_background_good.mean())


## Construct a deliberately bad but normalized model

A deterministic deformation is preferable to an accidentally undertrained classifier because its severity is reproducible and easy to interpret. The misspecified process ratios are

$$
r_S^{\rm bad}=(1-\kappa)+\kappa r_B^{\rm good},\qquad
r_B^{\rm bad}=(1-\kappa)+\kappa r_S^{\rm good},
$$

with $\kappa=0.25$. The labels are crossed and the distinction between the processes is diluted toward the reference density. The two ratios remain positive and normalized, so they define a valid statistical model even though it is intentionally unlike the original one.


In [ ]:
ratio_signal_bad_unnormalized = (
    1.0 - DEFORMATION_DILUTION
    + DEFORMATION_DILUTION * ratio_background_good
)
ratio_background_bad_unnormalized = (
    1.0 - DEFORMATION_DILUTION
    + DEFORMATION_DILUTION * ratio_signal_good
)
BAD_RATIO_NORMALIZATION = {
    "signal": float(ratio_signal_bad_unnormalized.mean()),
    "background": float(ratio_background_bad_unnormalized.mean()),
}
ratio_signal_bad = (
    ratio_signal_bad_unnormalized / BAD_RATIO_NORMALIZATION["signal"]
)
ratio_background_bad = (
    ratio_background_bad_unnormalized / BAD_RATIO_NORMALIZATION["background"]
)
weight_signal_bad = ratio_signal_bad / N_REFERENCE_EVENTS
weight_background_bad = ratio_background_bad / N_REFERENCE_EVENTS

log_process_ratio_good = np.log(ratio_signal_good / ratio_background_good)
log_process_ratio_bad = np.log(ratio_signal_bad / ratio_background_bad)
correlation = np.corrcoef(log_process_ratio_good, log_process_ratio_bad)[0, 1]
print(f"Good/bad log-ratio correlation: {correlation:.6f}")
print("Bad ratio means:", ratio_signal_bad.mean(), ratio_background_bad.mean())

rng = np.random.default_rng(SEED + 701)
plot_indices = rng.choice(
    N_REFERENCE_EVENTS, size=min(60_000, N_REFERENCE_EVENTS), replace=False
)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
axes[0].hexbin(
    log_process_ratio_good[plot_indices], log_process_ratio_bad[plot_indices],
    gridsize=65, bins="log", mincnt=1, cmap="viridis",
)
limit = np.quantile(
    np.abs(np.concatenate([log_process_ratio_good[plot_indices], log_process_ratio_bad[plot_indices]])),
    0.995,
)
axes[0].plot([-limit, limit], [-limit, limit], "k--", lw=1.3)
axes[0].set_xlim(-limit, limit)
axes[0].set_ylim(-limit, limit)
axes[0].set_xlabel(r"Original $\log(r_S/r_B)$")
axes[0].set_ylabel(r"Misspecified $\log(r_S/r_B)$")
axes[0].set_title("Crossed and diluted ratios")
axes[0].text(0.04, 0.95, rf"$\rho={correlation:.3f}$", transform=axes[0].transAxes, va="top")
bins = np.linspace(-limit, limit, 70)
axes[1].hist(log_process_ratio_good, bins=bins, density=True, histtype="step", lw=2, label="Original")
axes[1].hist(log_process_ratio_bad, bins=bins, density=True, histtype="step", lw=2, label="Misspecified")
axes[1].set_xlabel(r"$\log(r_S/r_B)$ on the reference")
axes[1].set_ylabel("Density")
axes[1].set_title("Loss of process separation")
axes[1].legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / "ratio_deformation.png", dpi=160)
export_exercise7_figure(fig, "ratio_deformation")
plt.show()


## Model-relative Asimov closure survives the deformation

For either pair of normalized ratios, the Asimov intensity is $H_A=\mu_A\lambda_S r_S+\lambda_B r_B$. The finite weighted scan is constructed from those same ratios. At $\mu_A=1$, its score is

$$
2\lambda_S\left[1-\frac{1}{M}\sum_m r_S(x_m)\right]=0.
$$

No statement about the physical correctness of $r_S$ or $r_B$ is used in this cancellation. The bad model must therefore fit its own Asimov sample at one. Its scan should nevertheless be shallower because most of the useful shape difference has been erased.


In [ ]:
def finite_asimov_scan(ratio_signal, ratio_background, mu_values):
    ratio_signal = np.asarray(ratio_signal, dtype=np.float64)
    ratio_background = np.asarray(ratio_background, dtype=np.float64)
    ratio_signal = ratio_signal / ratio_signal.mean()
    ratio_background = ratio_background / ratio_background.mean()
    h_asimov = (
        ASIMOV_MU_TRUE * LAM_SIG * ratio_signal
        + LAM_BKG * ratio_background
    )
    scan = []
    for mu in np.asarray(mu_values, dtype=np.float64):
        h_mu = mu * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
        statistic = 2.0 * (
            (mu - ASIMOV_MU_TRUE) * LAM_SIG
            + np.mean(h_asimov * np.log(h_asimov / h_mu))
        )
        scan.append(max(0.0, float(statistic)))
    score = 2.0 * LAM_SIG * (1.0 - ratio_signal.mean())
    return np.asarray(scan), float(score)


GOOD_ASIMOV_SCAN, GOOD_ASIMOV_SCORE = finite_asimov_scan(
    ratio_signal_good, ratio_background_good, MU_SCAN
)
BAD_ASIMOV_SCAN, BAD_ASIMOV_SCORE = finite_asimov_scan(
    ratio_signal_bad, ratio_background_bad, MU_SCAN
)
truth_index = int(np.argmin(np.abs(MU_SCAN - ASIMOV_MU_TRUE)))
zero_index = int(np.argmin(np.abs(MU_SCAN)))
GOOD_Q_ZERO = float(GOOD_ASIMOV_SCAN[zero_index])
BAD_Q_ZERO = float(BAD_ASIMOV_SCAN[zero_index])
GOOD_SIGMA = ASIMOV_MU_TRUE / np.sqrt(GOOD_Q_ZERO)
BAD_SIGMA = ASIMOV_MU_TRUE / np.sqrt(BAD_Q_ZERO)

print(f"Original Asimov score at mu_A:     {GOOD_ASIMOV_SCORE:+.3e}")
print(f"Misspecified Asimov score at mu_A: {BAD_ASIMOV_SCORE:+.3e}")
print(f"Original q_0,A / sigma_A:     {GOOD_Q_ZERO:.6f} / {GOOD_SIGMA:.6f}")
print(f"Misspecified q_0,A / sigma_A: {BAD_Q_ZERO:.6f} / {BAD_SIGMA:.6f}")
print(f"Relative information retained: {BAD_Q_ZERO / GOOD_Q_ZERO:.2%}")

fig, ax = plt.subplots(figsize=(7.0, 5.0))
ax.plot(MU_SCAN, GOOD_ASIMOV_SCAN, lw=2.3, label="Original Exercise 5 model")
ax.plot(MU_SCAN, BAD_ASIMOV_SCAN, lw=2.3, label="Misspecified model")
ax.axvline(ASIMOV_MU_TRUE, color="0.4", ls="--", lw=1.2)
ax.set_xlabel(r"$\mu$")
ax.set_ylabel(r"$t_A(\mu)$")
ax.set_title("Exact finite-sample closure, different sensitivity")
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / "asimov_scan_misspecification.png", dpi=160)
export_exercise7_figure(fig, "asimov_scan_misspecification")
plt.show()


## Coverage is model-relative

Two ensembles are now fitted with the same misspecified likelihood:

* **Misspecified-hNDE toys** are generated with the misspecified process weights. Their agreement with the hNDE Asimov/Wald prediction tests whether the Asimov construction, likelihood, and toy generator are internally consistent.
* **Simulator-based toys** are generated from a fresh bank of the latent Gaussian-mixture and detector-response simulator, passed through the fixed PRESEL selection, and evaluated with the frozen networks. They test the same likelihood against the external data-generating process.

The event likelihood depends only on the bad-model scalar score $q=\lambda_S r_S^{\rm bad}/(\lambda_B r_B^{\rm bad})$. It is compressed into 512 cells, as in the large Exercise 5 toy study. The bin likelihood ratio is derived from the misspecified model probabilities, so the compressed inference model is exactly self-consistent. Five million independently simulated selected events per component determine the external bin probabilities. Half of the hNDE toys are used to determine empirical critical values, and the other half are retained as an independent coverage sample.


In [ ]:
bad_event_q = (
    LAM_SIG / LAM_BKG * ratio_signal_bad / ratio_background_bad
)
bad_log_q = np.log(np.maximum(bad_event_q, np.finfo(np.float64).tiny))
lower, upper = np.quantile(bad_log_q, [1.0e-6, 1.0 - 1.0e-6])
log_q_edges = np.concatenate(
    ([-np.inf], np.linspace(lower, upper, TOY_Q_BINS - 1), [np.inf])
)


def score_probabilities(event_weights):
    probability = np.histogram(
        bad_log_q, bins=log_q_edges, weights=event_weights
    )[0].astype(np.float64)
    probability = probability + 1.0e-15
    return probability / probability.sum()


BAD_SIGNAL_PROBABILITY = score_probabilities(weight_signal_bad)
BAD_BACKGROUND_PROBABILITY = score_probabilities(weight_background_bad)
COMPRESSED_BAD_Q = (
    LAM_SIG / LAM_BKG
    * BAD_SIGNAL_PROBABILITY / BAD_BACKGROUND_PROBABILITY
)

bad_expected_counts = (
    ASIMOV_MU_TRUE * LAM_SIG * BAD_SIGNAL_PROBABILITY
    + LAM_BKG * BAD_BACKGROUND_PROBABILITY
)
COMPRESSED_BAD_Q_ZERO = 2.0 * (
    -ASIMOV_MU_TRUE * LAM_SIG
    - np.sum(
        bad_expected_counts
        * (np.log1p(0.0 * COMPRESSED_BAD_Q)
           - np.log1p(ASIMOV_MU_TRUE * COMPRESSED_BAD_Q))
    )
)
compression_difference = COMPRESSED_BAD_Q_ZERO / BAD_Q_ZERO - 1.0
TOY_ASIMOV_Q_ZERO = float(COMPRESSED_BAD_Q_ZERO)
TOY_ASIMOV_SIGMA = ASIMOV_MU_TRUE / np.sqrt(TOY_ASIMOV_Q_ZERO)
print(f"Full misspecified q_0,A:       {BAD_Q_ZERO:.8f}")
print(f"Compressed misspecified q_0,A: {COMPRESSED_BAD_Q_ZERO:.8f}")
print(f"Relative compression difference: {compression_difference:+.3%}")
if abs(compression_difference) > 5.0e-3:
    raise RuntimeError("Increase TOY_Q_BINS: q_0,A changed by more than 0.5%.")


### Build a genuinely external simulator ensemble

The earlier version used toys from the original Exercise 5 hNDE as a proxy for the physical process. Here that proxy is replaced by fresh simulator events. The simulator bank is independent of the samples used to train the reference flow and ratio estimators. Each selected event is evaluated once with the frozen networks, transformed by the same deterministic misspecification, and reduced to $q_{\rm bad}(x)$. The bank is cached, so rerunning the notebook repeats only the inexpensive Poisson toys.

The nominal post-selection yields remain fixed at $\lambda_S$ and $\lambda_B$. The comparison therefore isolates shape misspecification rather than adding an acceptance-rate difference. In the final figure, the simulator error bars show the Poisson uncertainty of the 100,000 toy counts; uncertainty from the much larger cached event bank is not shown.


In [ ]:
def simulate_reconstructed_mixture(components, n_events, rng):
    """Run the latent Gaussian-mixture and detector-response simulator."""
    n_events = int(n_events)
    fractions = np.asarray([component[0] for component in components], dtype=float)
    fractions /= fractions.sum()
    component_index = rng.choice(len(components), size=n_events, p=fractions)

    latent = np.empty((n_events, N_DIM), dtype=np.float64)
    for index, (_, mean, covariance) in enumerate(components):
        mask = component_index == index
        if np.any(mask):
            latent[mask] = rng.multivariate_normal(
                np.asarray(mean, dtype=float),
                np.asarray(covariance, dtype=float),
                size=int(mask.sum()),
            )

    response_scale, response_resolution = smearing_parameters()
    reconstructed = (
        np.asarray(response_scale, dtype=np.float64)[None, :] * latent
        + rng.normal(
            loc=0.0,
            scale=np.asarray(response_resolution, dtype=np.float64)[None, :],
            size=latent.shape,
        )
    )
    return pd.DataFrame(
        reconstructed.astype(np.float32, copy=False),
        columns=FEATURES,
    )


def evaluate_bad_model_q(values):
    """Evaluate the frozen, deliberately misspecified likelihood ratio."""
    signal_good = (
        evaluate_ratio("signal", values)
        / GOOD_RATIO_NORMALIZATION["signal"]
    )
    background_good = (
        evaluate_ratio("background", values)
        / GOOD_RATIO_NORMALIZATION["background"]
    )
    signal_bad = (
        1.0 - DEFORMATION_DILUTION
        + DEFORMATION_DILUTION * background_good
    ) / BAD_RATIO_NORMALIZATION["signal"]
    background_bad = (
        1.0 - DEFORMATION_DILUTION
        + DEFORMATION_DILUTION * signal_good
    ) / BAD_RATIO_NORMALIZATION["background"]
    log_q = (
        np.log(LAM_SIG / LAM_BKG)
        + np.log(signal_bad)
        - np.log(background_bad)
    )
    return np.exp(np.clip(log_q, -80.0, 80.0))


SIMULATOR_BANK_METADATA = {
    "presel_ratio_cut": PRESEL_RATIO_CUT,
    "deformation_dilution": DEFORMATION_DILUTION,
    "good_normalization_signal": GOOD_RATIO_NORMALIZATION["signal"],
    "good_normalization_background": GOOD_RATIO_NORMALIZATION["background"],
    "bad_normalization_signal": BAD_RATIO_NORMALIZATION["signal"],
    "bad_normalization_background": BAD_RATIO_NORMALIZATION["background"],
    "lambda_signal": LAM_SIG,
    "lambda_background": LAM_BKG,
}


def simulator_bank_path(sample_name, seed):
    return SIMULATOR_BANK_DIR / (
        f"{sample_name}_selected_bad_q_"
        f"{SIMULATOR_BANK_SELECTED_EVENTS:d}_seed{int(seed):d}.npz"
    )


def simulator_bank_is_compatible(payload):
    required = {"q"} | set(SIMULATOR_BANK_METADATA)
    if not required.issubset(payload.files):
        return False
    checks = [len(payload["q"]) == SIMULATOR_BANK_SELECTED_EVENTS]
    checks.extend(
        np.isclose(
            float(payload[key]),
            float(value),
            rtol=1.0e-12,
            atol=1.0e-15,
        )
        for key, value in SIMULATOR_BANK_METADATA.items()
    )
    return bool(np.all(checks))


def generate_selected_simulator_q_bank(sample_name, seed):
    components_by_sample = {
        "signal": signal_components(),
        "background": background_components(),
    }
    if sample_name not in components_by_sample:
        raise KeyError(f"Unknown simulator sample: {sample_name}")

    rng = np.random.default_rng(int(seed))
    q_chunks = []
    n_kept = 0
    n_generated = 0
    n_passed = 0
    report_step = max(1, SIMULATOR_BANK_SELECTED_EVENTS // 10)
    next_report = report_step

    while n_kept < SIMULATOR_BANK_SELECTED_EVENTS:
        generated = simulate_reconstructed_mixture(
            components_by_sample[sample_name],
            SIMULATOR_BANK_GENERATION_BATCH_SIZE,
            rng,
        )
        passes = evaluate_PRESEL_ratio(generated) >= PRESEL_RATIO_CUT
        selected = generated.loc[passes, FEATURES].reset_index(drop=True)
        n_generated += len(generated)
        n_passed += len(selected)

        if len(selected):
            q_batch = evaluate_bad_model_q(selected)
            n_to_keep = min(
                len(q_batch),
                SIMULATOR_BANK_SELECTED_EVENTS - n_kept,
            )
            q_chunks.append(q_batch[:n_to_keep].astype(np.float64, copy=False))
            n_kept += n_to_keep

        if n_kept >= next_report or n_kept == SIMULATOR_BANK_SELECTED_EVENTS:
            print(
                f"{sample_name:10s}: retained "
                f"{n_kept:,}/{SIMULATOR_BANK_SELECTED_EVENTS:,} selected events "
                f"after generating {n_generated:,}"
            )
            while next_report <= n_kept:
                next_report += report_step

        del generated, selected, passes
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    q_values = np.concatenate(q_chunks)[:SIMULATOR_BANK_SELECTED_EVENTS]
    print(
        f"{sample_name:10s}: simulator PRESEL acceptance = "
        f"{n_passed / n_generated:.3%}"
    )
    return q_values


def load_or_generate_simulator_q_bank(sample_name, seed):
    path = simulator_bank_path(sample_name, seed)
    if path.exists() and not SIMULATOR_BANK_FORCE_REBUILD:
        with np.load(path, allow_pickle=False) as payload:
            if simulator_bank_is_compatible(payload):
                q_values = np.asarray(payload["q"], dtype=np.float64)
                print(f"Loaded {len(q_values):,} {sample_name} q values from {path}")
                return q_values
        print(f"Ignoring incompatible cached bank: {path}")

    q_values = generate_selected_simulator_q_bank(sample_name, seed)
    np.savez(
        path,
        q=q_values,
        **{
            key: np.asarray(value)
            for key, value in SIMULATOR_BANK_METADATA.items()
        },
    )
    print(f"Saved simulator bank to {path}")
    return q_values


SIMULATOR_Q_SIGNAL = load_or_generate_simulator_q_bank(
    "signal", SIMULATOR_BANK_SEED + 1
)
SIMULATOR_Q_BACKGROUND = load_or_generate_simulator_q_bank(
    "background", SIMULATOR_BANK_SEED + 2
)


def simulator_probability_in_bad_q_bins(q_values):
    log_q = np.log(np.maximum(q_values, np.finfo(np.float64).tiny))
    probability = np.histogram(log_q, bins=log_q_edges)[0].astype(np.float64)
    probability = probability + 1.0e-15
    return probability / probability.sum()


SIMULATOR_SIGNAL_PROBABILITY = simulator_probability_in_bad_q_bins(
    SIMULATOR_Q_SIGNAL
)
SIMULATOR_BACKGROUND_PROBABILITY = simulator_probability_in_bad_q_bins(
    SIMULATOR_Q_BACKGROUND
)


In [ ]:
COMPRESSED_BAD_Q_JAX = jnp.asarray(COMPRESSED_BAD_Q)


@jax.jit
def fit_bad_model_batch(counts, test_mu):
    counts = jnp.asarray(counts, dtype=jnp.float64)
    q_values = COMPRESSED_BAD_Q_JAX
    initial_mu = jnp.clip(
        (jnp.sum(counts, axis=1) - LAM_BKG) / LAM_SIG, 0.0, TOY_MU_MAX
    )
    score_at_zero = LAM_SIG - jnp.sum(counts * q_values, axis=1)

    def newton_step(_, mu):
        response = q_values / (1.0 + mu[:, None] * q_values)
        score = LAM_SIG - jnp.sum(counts * response, axis=1)
        information = jnp.sum(counts * response**2, axis=1)
        step = jnp.clip(score / jnp.maximum(information, 1.0e-12), -2.0, 2.0)
        return jnp.clip(mu - step, 0.0, TOY_MU_MAX)

    mu_hat = jax.lax.fori_loop(0, TOY_NEWTON_STEPS, newton_step, initial_mu)
    mu_hat = jnp.where(score_at_zero >= 0.0, 0.0, mu_hat)

    def statistic(mu_test):
        value = 2.0 * (
            (mu_test - mu_hat) * LAM_SIG
            - jnp.sum(
                counts
                * (
                    jnp.log1p(mu_test * q_values)
                    - jnp.log1p(mu_hat[:, None] * q_values)
                ),
                axis=1,
            )
        )
        return jnp.maximum(value, 0.0)

    response_at_test = q_values / (1.0 + test_mu * q_values)
    information = jnp.sum(counts * response_at_test**2, axis=1)
    return mu_hat, statistic(test_mu), statistic(0.0), information


def run_toy_ensemble(
    generator_name, signal_probability, background_probability, seed
):
    rng = np.random.default_rng(seed)
    signal_means = ASIMOV_MU_TRUE * LAM_SIG * signal_probability
    background_means = LAM_BKG * background_probability
    chunks = []
    n_batches = int(np.ceil(N_TOYS / TOY_BATCH_SIZE))
    for batch_index, start in enumerate(range(0, N_TOYS, TOY_BATCH_SIZE)):
        batch_size = min(TOY_BATCH_SIZE, N_TOYS - start)
        signal_counts = rng.poisson(
            signal_means, size=(batch_size, TOY_Q_BINS)
        )
        background_counts = rng.poisson(
            background_means, size=(batch_size, TOY_Q_BINS)
        )
        counts = signal_counts + background_counts
        mu_hat, t_mu, q_zero, information = fit_bad_model_batch(
            counts, ASIMOV_MU_TRUE
        )
        chunks.append(
            pd.DataFrame(
                {
                    "generator": generator_name,
                    "toy": np.arange(start, start + batch_size),
                    "n_events": counts.sum(axis=1),
                    "mu_hat": np.asarray(mu_hat),
                    "t_mu": np.asarray(t_mu),
                    "q_zero": np.asarray(q_zero),
                    "information": np.asarray(information),
                }
            )
        )
        if (batch_index + 1) % max(1, n_batches // 10) == 0:
            print(
                f"{generator_name}: completed {start + batch_size:,}/{N_TOYS:,} toys"
            )
    return pd.concat(chunks, ignore_index=True)


SIMULATOR_ASIMOV_EXPECTED_COUNTS = (
    ASIMOV_MU_TRUE * LAM_SIG * SIMULATOR_SIGNAL_PROBABILITY
    + LAM_BKG * SIMULATOR_BACKGROUND_PROBABILITY
)
(
    simulator_asimov_mu_hat,
    simulator_asimov_t_at_truth,
    simulator_asimov_q_zero,
    simulator_asimov_information,
) = fit_bad_model_batch(
    SIMULATOR_ASIMOV_EXPECTED_COUNTS[None, :],
    ASIMOV_MU_TRUE,
)
SIMULATOR_ASIMOV_MU_HAT = float(np.asarray(simulator_asimov_mu_hat)[0])
SIMULATOR_ASIMOV_T_AT_TRUTH = float(
    np.asarray(simulator_asimov_t_at_truth)[0]
)
SIMULATOR_ASIMOV_Q_ZERO = float(np.asarray(simulator_asimov_q_zero)[0])
SIMULATOR_ASIMOV_INFORMATION = float(
    np.asarray(simulator_asimov_information)[0]
)
print("\nExternal simulator-Asimov diagnostics")
print(f"  injected mu_A:                    {ASIMOV_MU_TRUE:.8f}")
print(f"  fitted pseudo-true mu:            {SIMULATOR_ASIMOV_MU_HAT:.8f}")
print(f"  t_mu at injected mu_A:            {SIMULATOR_ASIMOV_T_AT_TRUTH:.8f}")
print(f"  q_0,A from simulator expectation: {SIMULATOR_ASIMOV_Q_ZERO:.8f}")

model_toys = run_toy_ensemble(
    MODEL_GENERATOR_LABEL,
    BAD_SIGNAL_PROBABILITY,
    BAD_BACKGROUND_PROBABILITY,
    SEED + 800,
)
simulator_toys = run_toy_ensemble(
    SIMULATOR_GENERATOR_LABEL,
    SIMULATOR_SIGNAL_PROBABILITY,
    SIMULATOR_BACKGROUND_PROBABILITY,
    SEED + 801,
)
toy_results = pd.concat([model_toys, simulator_toys], ignore_index=True)


In [ ]:
asymptotic_critical_values = chi2.ppf(COVERAGE_LEVELS, df=1)
model_calibration_toys = model_toys.loc[model_toys["toy"] % 2 == 0]
model_evaluation_toys = model_toys.loc[model_toys["toy"] % 2 == 1]
calibrated_critical_values = np.quantile(
    model_calibration_toys["t_mu"], COVERAGE_LEVELS
)
coverage_rows = []
for generator_name, group in toy_results.groupby("generator", sort=False):
    if generator_name == MODEL_GENERATOR_LABEL:
        group = model_evaluation_toys
    for level, asymptotic_critical, calibrated_critical in zip(
        COVERAGE_LEVELS, asymptotic_critical_values, calibrated_critical_values
    ):
        coverage_rows.append(
            {
                "generator": generator_name,
                "nominal": level,
                "asymptotic_coverage": float(
                    np.mean(group["t_mu"] <= asymptotic_critical)
                ),
                "hnde_toy_calibrated_coverage": float(
                    np.mean(group["t_mu"] <= calibrated_critical)
                ),
                "asymptotic_critical": asymptotic_critical,
                "hnde_toy_critical": calibrated_critical,
            }
        )
coverage_table = pd.DataFrame(coverage_rows)
display(coverage_table)

summary = toy_results.groupby("generator")["mu_hat"].agg(["mean", "std"])
summary["rms_about_truth"] = toy_results.groupby("generator")["mu_hat"].apply(
    lambda values: np.sqrt(np.mean((values - ASIMOV_MU_TRUE) ** 2))
)
summary["boundary_fraction"] = toy_results.groupby("generator")["mu_hat"].apply(
    lambda values: np.mean(values <= 1.0e-12)
)
display(summary)
print(f"Misspecified-hNDE Asimov sigma: {TOY_ASIMOV_SIGMA:.6f}")


def histogram_probability_and_error(values, edges):
    counts = np.histogram(values, bins=edges)[0].astype(np.float64)
    probability = counts / len(values)
    error = np.sqrt(counts) / len(values)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return probability, error, centers, counts


fig, axes = plt.subplots(1, 2, figsize=(12.0, 5.2))

# The hNDE Asimov construction predicts a bounded Gaussian for mu_hat.
mu_upper = max(
    float(np.quantile(toy_results["mu_hat"], 0.999)) * 1.001,
    ASIMOV_MU_TRUE + 4.0 * TOY_ASIMOV_SIGMA,
    1.25,
)
mu_edges = np.linspace(0.0, mu_upper, 65)
model_mu_probability, _, _, _ = histogram_probability_and_error(
    model_toys["mu_hat"], mu_edges
)
(
    simulator_mu_probability,
    simulator_mu_error,
    mu_centers,
    simulator_mu_counts,
) = histogram_probability_and_error(simulator_toys["mu_hat"], mu_edges)
mu_wald_cdf = norm.cdf(
    (mu_edges - ASIMOV_MU_TRUE) / TOY_ASIMOV_SIGMA
)
mu_wald_probability = np.diff(mu_wald_cdf)
mu_wald_probability[0] += mu_wald_cdf[0]

axes[0].stairs(
    model_mu_probability,
    mu_edges,
    color="C0",
    lw=2.3,
    label=f"Misspecified-hNDE toys ({len(model_toys):,})",
)
valid_mu = simulator_mu_counts > 0
axes[0].errorbar(
    mu_centers[valid_mu],
    simulator_mu_probability[valid_mu],
    yerr=simulator_mu_error[valid_mu],
    fmt="o",
    ms=3.5,
    capsize=1.5,
    color="black",
)
axes[0].plot(
    [], [], color="black", marker="o", ls="none", ms=3.5,
    label=f"Simulator-based toys ({len(simulator_toys):,})",
)
axes[0].stairs(
    mu_wald_probability,
    mu_edges,
    color="C3",
    lw=2.0,
    ls="--",
    label=rf"hNDE Asimov/Wald ($\sigma_A={TOY_ASIMOV_SIGMA:.3f}$)",
)
axes[0].axvline(ASIMOV_MU_TRUE, color="0.45", ls=":", lw=1.3)
axes[0].axvline(
    SIMULATOR_ASIMOV_MU_HAT,
    color="black",
    ls="-.",
    lw=1.2,
    label=rf"Simulator pseudo-true $\mu={SIMULATOR_ASIMOV_MU_HAT:.3f}$",
)
axes[0].set_xlabel(r"$\hat\mu$ from the misspecified likelihood")
axes[0].set_ylabel("Probability per bin")
axes[0].set_title("Estimator closure")
axes[0].legend(fontsize=8, loc="upper right")

# The Asimov q_0,A fixes the boundary-aware Cowan prediction.
q_upper = max(
    float(np.quantile(toy_results["q_zero"], 0.9995)) * 1.001,
    float(ncx2.ppf(0.9995, df=1, nc=TOY_ASIMOV_Q_ZERO)),
    1.0,
)
q_edges = np.linspace(0.0, q_upper, 65)
model_q_probability, _, _, _ = histogram_probability_and_error(
    model_toys["q_zero"], q_edges
)
(
    simulator_q_probability,
    simulator_q_error,
    q_centers,
    simulator_q_counts,
) = histogram_probability_and_error(simulator_toys["q_zero"], q_edges)
q_boundary_cdf = norm.cdf(
    np.sqrt(q_edges) - np.sqrt(TOY_ASIMOV_Q_ZERO)
)
q_boundary_probability = np.diff(q_boundary_cdf)
q_boundary_probability[0] += q_boundary_cdf[0]

axes[1].stairs(
    model_q_probability,
    q_edges,
    color="C0",
    lw=2.3,
    label=f"Misspecified-hNDE toys ({len(model_toys):,})",
)
valid_q = simulator_q_counts > 0
axes[1].errorbar(
    q_centers[valid_q],
    simulator_q_probability[valid_q],
    yerr=simulator_q_error[valid_q],
    fmt="o",
    ms=3.5,
    capsize=1.5,
    color="black",
)
axes[1].plot(
    [], [], color="black", marker="o", ls="none", ms=3.5,
    label=f"Simulator-based toys ({len(simulator_toys):,})",
)
axes[1].stairs(
    q_boundary_probability,
    q_edges,
    color="C3",
    lw=2.0,
    ls="--",
    label="hNDE Asimov/Cowan prediction",
)
axes[1].set_yscale("log")
axes[1].set_ylim(bottom=0.5 / max(len(model_toys), len(simulator_toys)))
axes[1].set_xlabel(r"$q_0$")
axes[1].set_ylabel("Probability per bin")
axes[1].set_title(r"Discovery statistic under $\mu_{\rm true}=1$")
axes[1].legend(fontsize=8, loc="upper right")

fig.tight_layout()
fig.savefig(PLOT_DIR / "coverage_misspecification.png", dpi=160)
export_exercise7_figure(fig, "coverage_misspecification")
plt.show()


## Interpretation

The exact minimum of the misspecified Asimov scan is an algebraic normalization result. It establishes that the Asimov sample, likelihood, and process yields describe one internally coherent approximate model. The misspecified-hNDE toy ensemble makes that internal statement visible: its $\hat\mu$ distribution follows the bounded Gaussian predicted by the hNDE Asimov construction, and its $q_0$ distribution follows the corresponding boundary-aware Cowan prediction.

The independent simulator ensemble answers the physically different question. Its events are not distributed according to the misspecified hNDE likelihood being fitted. The displacement of its $\hat\mu$ distribution from the hNDE Asimov/Wald curve, and the corresponding distortion of $q_0$, are genuine external model-misspecification effects. The coverage table makes the same point directly with $t_{\mu_{\rm true}}$: neither exact hNDE Asimov closure nor critical values calibrated with hNDE toys can guarantee coverage for simulator-generated data.

This is the central lesson of the exercise: the Asimov construction can describe pseudo-experiments from an approximate model extremely well while inference for simulator-generated data is miscalibrated. Finite-sample ratio normalization prevents an additional internal inconsistency; it does not make the learned model exact. Independent simulator checks at representative parameter points, or simulator-based calibration when needed, remain necessary for coverage with respect to the physical data-generating process.


In [ ]:
assert abs(BAD_ASIMOV_SCORE) < 1.0e-10
assert int(np.argmin(BAD_ASIMOV_SCAN)) == truth_index

model_95 = coverage_table.loc[
    (coverage_table["generator"] == MODEL_GENERATOR_LABEL)
    & np.isclose(coverage_table["nominal"], 0.95),
    "hnde_toy_calibrated_coverage",
].iloc[0]
simulator_95_asymptotic = coverage_table.loc[
    (coverage_table["generator"] == SIMULATOR_GENERATOR_LABEL)
    & np.isclose(coverage_table["nominal"], 0.95),
    "asymptotic_coverage",
].iloc[0]
simulator_95_hnde_calibrated = coverage_table.loc[
    (coverage_table["generator"] == SIMULATOR_GENERATOR_LABEL)
    & np.isclose(coverage_table["nominal"], 0.95),
    "hnde_toy_calibrated_coverage",
].iloc[0]

print(f"Independent hNDE-toy-calibrated 95% coverage: {model_95:.4%}")
print(f"Simulator coverage with asymptotic threshold: {simulator_95_asymptotic:.4%}")
print(
    "Simulator coverage with hNDE-toy threshold: "
    f"{simulator_95_hnde_calibrated:.4%}"
)
print("PASS: the misspecified hNDE is internally calibrated on held-out toys.")
print("The simulator rows quantify the external coverage not guaranteed by closure.")


## Suggested exercises

1. Vary `DEFORMATION_DILUTION` from one to zero and plot expected information and coverage.
2. Remove the signal/background crossing while retaining the dilution. Determine whether the simulator-based coverage improves.
3. Calibrate critical values with simulator-based toys and compare them with the misspecified-hNDE critical values. Which coverage statement is being made in each case?
4. Add nuisance parameters and determine which finite-sample score cancellations survive when shape parameters, rather than only yields, are varied.
